# Gestión de Datos y Validación de Modelos

**Objetivos**
1. Buscar y descargar datasets desde Kaggle y otras fuentes.
2. Etiquetado de datos (manual y asistido).
3. Validación cruzada: ejercicios prácticos y comparativos extensos.


## 1. Búsqueda de Datasets

### 1.1 Teoría
- Importancia de un buen dataset.
- Fuentes: Kaggle, UCI, Google Dataset Search, AWS Open Data.
- Licencias y descripción de variables.


In [5]:
!pip install kaggle --quiet

#### Ejemplo: API de Kaggle

In [6]:
from kaggle.api.kaggle_api_extended import KaggleApi
import pandas as pd

# api = KaggleApi()
# api.authenticate()

# # Descargar Titanic
# api.dataset_download_files('heptapod/titanic', path='data/titanic', unzip=True)
# df_titanic = pd.read_csv('data/titanic/train.csv')
# df_titanic.head()

#### Ejemplo: UCI Repository

In [7]:
import pandas as pd
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'
columns = ['sepal_length','sepal_width','petal_length','petal_width','class']
iris = pd.read_csv(url, header=None, names=columns)
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


**Ejercicio 1**: Busca y descarga un dataset usando Python de una fuente distinta a Kaggle o UCI. Carga un DataFrame y muestra sus primeras filas.

## 2. Etiquetado de Datos

### 2.1 Teoría
- Manual vs asistido.
- Herramientas: Label Studio, Prodigy, VIA.
- Tipos de etiquetas: clasificación, bounding boxes, segmentación.


In [8]:
# Ejemplo de etiquetado manual
data = [
    {'texto': 'Me encanta este producto', 'sentimiento': None},
    {'texto': 'El servicio fue horrible', 'sentimiento': None},
]
import pandas as pd
df_labels = pd.DataFrame(data)
df_labels

,texto,sentimiento
0,Me encanta este producto,None
1,El servicio fue horrible,None


In [9]:
# Asignar etiquetas manualmente
df_labels.loc[0, 'sentimiento'] = 'positivo'
df_labels.loc[1, 'sentimiento'] = 'negativo'
df_labels

,texto,sentimiento
0,Me encanta este producto,positivo
1,El servicio fue horrible,negativo


**Ejercicio 2**: Crea un mini-dataset de frases y etiqueta una clase de tu elección. Exporta a CSV.

## 3. Validación Cruzada (Extenso)

### 3.1 Teoría
- K-Fold, StratifiedKFold, Leave-One-Out, RepeatedKFold, TimeSeriesSplit.
- Nested Cross-Validation y selección de hiperparámetros.
- cross_val_score vs cross_val_predict.


In [10]:
from sklearn.datasets import load_iris, load_digits
from sklearn.model_selection import (KFold, StratifiedKFold, LeaveOneOut,
                                     RepeatedKFold, TimeSeriesSplit, cross_val_score,
                                     cross_val_predict, GridSearchCV)
from sklearn.tree import DecisionTreeClassifier
import numpy as np


### 3.2 Ejemplo: KFold vs StratifiedKFold

In [11]:
X, y = load_iris(return_X_y=True)
clf = DecisionTreeClassifier(random_state=0)

# KFold
kf = KFold(n_splits=5, shuffle=True, random_state=0)
scores_kf = cross_val_score(clf, X, y, cv=kf)

# StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores_skf = cross_val_score(clf, X, y, cv=skf)

print('KFold mean accuracy:', np.mean(scores_kf))
print('StratifiedKFold mean accuracy:', np.mean(scores_skf))

KFold mean accuracy: 0.9466666666666667
StratifiedKFold mean accuracy: 0.9333333333333333


**Ejercicio 3**: Compara KFold y StratifiedKFold en el dataset Digits (`load_digits`). ¿Influye el balance de clases?

### 3.3 Leave-One-Out

In [12]:
loo = LeaveOneOut()
scores_loo = cross_val_score(clf, X, y, cv=loo)
print('Leave-One-Out accuracy:', np.mean(scores_loo))

Leave-One-Out accuracy: 0.9533333333333334


**Ejercicio 4**: Aplica Leave-One-Out a un dataset pequeño (p.ej., Iris vs una submuestra) y compara tiempos y resultados.

### 3.4 RepeatedKFold

In [13]:
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=0)
scores_rkf = cross_val_score(clf, X, y, cv=rkf)
print('RepeatedKFold mean accuracy:', np.mean(scores_rkf))

RepeatedKFold mean accuracy: 0.9555555555555556


**Ejercicio 5**: Ajusta el número de repeticiones y observa la varianza del puntaje.

### 3.5 Nested Cross-Validation con GridSearchCV

In [14]:
param_grid = {'max_depth': [2, 3, 4, None]}
inner_cv = KFold(n_splits=4, shuffle=True, random_state=0)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=0)
clf_gs = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=inner_cv)
nested_scores = cross_val_score(clf_gs, X, y, cv=outer_cv)
print('Nested CV accuracy:', np.mean(nested_scores))

Nested CV accuracy: 0.9333333333333333


**Ejercicio 6**: Extiende el GridSearch con otros parámetros (criterio, min_samples_split) y compara resultados.

### 3.6 cross_val_predict y evaluación visual

In [15]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = cross_val_predict(clf, X, y, cv=skf)
print(classification_report(y, y_pred))
conf_mat = confusion_matrix(y, y_pred)
print('Confusion matrix:\n', conf_mat)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       0.92      0.88      0.90        50
           2       0.88      0.92      0.90        50

    accuracy                           0.93       150
   macro avg       0.93      0.93      0.93       150
weighted avg       0.93      0.93      0.93       150

Confusion matrix:
 [[50  0  0]
 [ 0 44  6]
 [ 0  4 46]]


**Ejercicio 7**: Visualiza la matriz de confusión usando matplotlib y analiza los errores de clasificación.

## Conclusión

Este notebook ampliado cubre múltiples estrategias de validación cruzada, desde K-Fold básico hasta Nested CV y predicciones cruzadas.
¡Practica con diferentes modelos y datasets para fortalecer tu comprensión!
